# MLP Pipeline Smoke Test
This notebook imports `mlp.py`, builds a small multitask training pipeline, and verifies it runs on a small sample.

In [ ]:
from pathlib import Path
from sklearn.model_selection import train_test_split

from mlp import (
    set_global_seed,
    load_labels,
    verify_images_exist,
    create_tf_dataset,
    build_multitask_model,
    train_model,
    evaluate_model,
)

set_global_seed(42)

In [ ]:
labels = load_labels(Path('data/labels.csv'))
sample_df = labels.sample(n=min(100, len(labels)), random_state=42).reset_index(drop=True)
sample_df, missing_images = verify_images_exist(sample_df, Path('data/images'))
print(f'Sample rows: {len(sample_df)} | Missing images skipped: {len(missing_images)}')
assert len(sample_df) > 0, 'No valid sample rows with existing images'

In [ ]:
train_df, temp_df = train_test_split(sample_df, test_size=0.3, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

train_ds = create_tf_dataset(train_df, images_dir=Path('data/images'), batch_size=8, image_size=(128, 128), augment=True, shuffle=True, seed=42)
val_ds = create_tf_dataset(val_df, images_dir=Path('data/images'), batch_size=8, image_size=(128, 128), augment=False, shuffle=False, seed=42)
test_ds = create_tf_dataset(test_df, images_dir=Path('data/images'), batch_size=8, image_size=(128, 128), augment=False, shuffle=False, seed=42)

batch_images, batch_labels = next(iter(train_ds))
print('Image batch shape:', batch_images.shape)
print('Gender label shape:', batch_labels['gender'].shape)
print('Age label shape:', batch_labels['age'].shape)

In [ ]:
model = build_multitask_model(input_shape=(128, 128, 3))
history = train_model(model, train_ds, val_ds=val_ds, epochs=2, seed=42)
metrics = evaluate_model(model, test_ds)
metrics